# lambda, map, filter & reduce: functions small enough to fit on one line

**▶ 01 Core Python** · 02 Pandas · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies

`01_Python_Fundamentals/05_lambda_map_filter_and_reduce.ipynb`

---

### In one paragraph (no jargon)

A `lambda` is a function with no name, written inline: `lambda x: x * 1.18` is the same as a two-line `def` that returns `x * 1.18`. It exists for the moments when writing a whole named function would be overkill, mainly when you're handing a rule to *another* function. Those other functions are `map` (apply this rule to every item), `filter` (keep the items where this rule is True) and `reduce` (fold the whole list into a single value). Together they are the vocabulary of 'functional' data work, and they show up in pandas as `.apply()`, `.map()` and boolean masks.

### After this notebook you can

- Write a lambda for any one-line rule and call it like a normal function
- Use `map` and `filter` and convert their results back into a list
- Chain `map` and `filter` together, and know when a comprehension reads better
- Sort a list of tuples or dictionaries by any field using `key=lambda …`
- Use `reduce` for running totals, and `operator` functions instead of trivial lambdas


### What's inside

1. Part 1: Comprehension warm-up (Q1-Q5)
2. Part 2: lambda (Q6-Q10)
3. ⚡ operator, itemgetter and reduce
4. Part 3: map (Q11-Q15)
5. Part 4: filter (Q16-Q20)
6. Part 5: Combining them (Q21-Q35)
7. ⚡ The pure-Python → pandas translation table
8. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory, so **every cell below still runs**,
> which is handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

### Jargon buster

| Word you'll see | What it actually means |
|---|---|
| **variable** | A labelled box that holds a value. `price = 250` puts 250 in a box called `price`. |
| **list** `[ ]` | An ordered shopping list of values. You can add, remove and reorder items. |
| **tuple** `( )` | Like a list, but frozen: once made it cannot be changed. Good for fixed records. |
| **dictionary** `{key: value}` | A labelled lookup table, like a contacts app: look up "Ravi", get his number. |
| **index** | The position number of an item. Python counts from **0**, so the 1st item is at index 0. |
| **function** | A named recipe you can re-run with different ingredients. |
| **argument / parameter** | The ingredients you hand to a function. |
| **return** | The answer a function hands back to you. |
| **iterate / loop** | Do the same thing once for every item in a list. |
| **boolean** | A yes/no value: `True` or `False`. |
| **string** `'text'` | Text. Always wrapped in quotes. |
| **f-string** `f"{x}"` | A sentence with live values slotted in: the report-writing tool. |

In [1]:
# =============================================================================
# SETUP — run this cell first. Everything below depends only on this.
# =============================================================================
# WHAT: these are Python's own built-in toolboxes; nothing needs installing.
import math                     # square roots, rounding, constants
import statistics               # mean / median without pandas
from collections import Counter, defaultdict   # counting and grouping helpers
from functools import reduce    # folds a whole list down to one value

# -----------------------------------------------------------------------------
# Exam-safe replacement for input()
# -----------------------------------------------------------------------------
# WHY: a notebook that calls input() STOPS and waits for typing. If the examiner
#      clicks "Run All", it hangs. `ask()` behaves exactly like input() but plays
#      back a scripted answer instead, so the notebook always runs end-to-end.
# 🔧 CHANGE THIS: set INTERACTIVE = True if you WANT to type answers yourself.
INTERACTIVE = False
_scripted_answers = iter(['42', '7', '3', '15', '100', '5', '2', '9', '1', '0'])

def ask(prompt=''):
    """Stand-in for input() that never blocks a Run All."""
    if INTERACTIVE:
        return input(prompt)
    value = next(_scripted_answers, '0')
    print(f"{prompt}{value}    <- simulated keyboard input")
    return value

print("Setup complete — you can now run any cell below in any order.")

Setup complete — you can now run any cell below in any order.


## Part 1: Warm-up with comprehensions (Q1-Q5)

These first five repeat the comprehension pattern in a business setting, so you can see side by side what `map`/`filter` will do differently.

**Q1. You have a list of sales figures `[120, 300, 150, 400]`. How can you use list comprehension to calculate a 10% commission for each sales value??**

In [2]:
# Solution for Q1
sales = [120, 300, 150, 400]
commissions = [amount * 0.10 for amount in sales]  # 10% commission per sale
print(commissions)

[12.0, 30.0, 15.0, 40.0]


**Q2. Given a list of employee names `['Alice', 'Bob', 'Charlie']`, how can you use list comprehension to create email IDs by adding `@company.com`??**

In [3]:
# Solution for Q2
employees = ['Alice', 'Bob', 'Charlie']
employee_emails = [f"{name.lower()}@company.com" for name in employees]  # create lowercase email IDs
print(employee_emails)

['alice@company.com', 'bob@company.com', 'charlie@company.com']


**Q3. From a list of product prices `[50, 120, 300, 25]`, how can you use list comprehension to filter only those greater than 100??**

In [4]:
# Solution for Q3
prices = [50, 120, 300, 25]
premium_prices = [price for price in prices if price > 100]  # filter costs above ₹100
print(premium_prices)

[120, 300]


**Q4. If you have a list of quantities `[2, 4, 6, 8]`, how can you use list comprehension to generate their square values??**

In [5]:
# Solution for Q4
quantities = [2, 4, 6, 8]
squared_quantities = [qty ** 2 for qty in quantities]  # emphasise exponential growth
print(squared_quantities)

[4, 16, 36, 64]


**Q5. From a list of customer feedback words `['good', 'bad', 'excellent']`, how can you use list comprehension to convert each to uppercase??**

In [6]:
# Solution for Q5
feedback = ['good', 'bad', 'excellent']
shouted_feedback = [word.upper() for word in feedback]  # standardise to uppercase for dashboards
print(shouted_feedback)

['GOOD', 'BAD', 'EXCELLENT']


## Part 2: `lambda` (Q6-Q10)

```python
double = lambda x: x * 2      # is exactly the same as…
def double(x):
    return x * 2
```

Rules: everything after the colon is the return value, there is no `return` keyword, and it must be a **single expression**, with no `if:` blocks and no loops. (A conditional *expression* is fine: `lambda x: 'High' if x > 500 else 'Low'`.)

A lambda can take several arguments: `lambda price, disc: price * (1 - disc/100)`.

**Q6. How can you use a lambda function to calculate profit given revenue and cost??**

In [7]:
# Solution for Q6
profit = lambda revenue, cost: revenue - cost  # inline function with two arguments
print(f"Profit: {profit(85000, 62000)}")

Profit: 23000


**Q7. Write a lambda function to calculate discount price given original price and discount percentage??**

In [8]:
# Solution for Q7
discount_price = lambda price, discount: price * (1 - discount / 100)
print(f"Discounted price: {discount_price(2000, 15)}")

Discounted price: 1700.0


**Q8. How can you create a lambda function that checks if a transaction amount is above 500??**

In [9]:
# Solution for Q8
is_high_value = lambda amount: amount > 500  # returns boolean flag
print(is_high_value(750))

True


In [10]:
# --- Another valid way to write the same answer -------------------
# WHY show two? Examiners give credit for either. Pick whichever you
# can reproduce from memory under time pressure.
is_above_500 = lambda amount: amount > 500

# Example list of transactions
transactions = [200, 600, 450, 800]

# Apply the lambda function to the list using list comprehension
results = [is_above_500(t) for t in transactions]
print(results)

[False, True, False, True]


**Q9. Write a lambda function to find maximum of two sales amounts??**

In [11]:
# Solution for Q9
max_sales = lambda s1, s2: s1 if s1 > s2 else s2  # inline comparison
print(max_sales(48000, 52500))

52500


**Q10. How can you use lambda to calculate GST (18%) on a given bill amount??**

In [12]:
# Solution for Q10
calculate_gst = lambda bill: bill * 0.18  # compute 18% GST component
print(f"GST for ₹1500: {calculate_gst(1500)}")

GST for ₹1500: 270.0


### ⚡ Beyond the syllabus: `operator`: the named functions that replace trivial lambdas

`lambda x, y: x + y` is so common that Python ships it pre-written as `operator.add`. Using the named versions is faster (they're written in C) and signals fluency. `itemgetter` and `attrgetter` are the big wins: they replace `key=lambda t: t[1]` with something self-documenting.

In [13]:
from operator import add, mul, itemgetter
from functools import reduce

revenue = [1200, 900, 1500, 2100]

print("Sum via reduce+lambda:", reduce(lambda a, b: a + b, revenue))
print("Sum via reduce+add   :", reduce(add, revenue))      # same, faster, clearer
print("Product via mul      :", reduce(mul, [1, 2, 3, 4]))

sales = [('Alpha', 300), ('Beta', 100), ('Gamma', 200)]

print("\nSort by value, lambda    :", sorted(sales, key=lambda t: t[1]))
print("Sort by value, itemgetter:", sorted(sales, key=itemgetter(1)))
print("Top 2 by value           :", sorted(sales, key=itemgetter(1), reverse=True)[:2])

# Sorting on TWO keys at once — by value descending, then name alphabetically
staff = [('Ravi', 5), ('Meera', 7), ('Ali', 5)]
print("Two-key sort             :", sorted(staff, key=lambda t: (-t[1], t[0])))

Sum via reduce+lambda: 5700
Sum via reduce+add   : 5700
Product via mul      : 24

Sort by value, lambda    : [('Beta', 100), ('Gamma', 200), ('Alpha', 300)]
Sort by value, itemgetter: [('Beta', 100), ('Gamma', 200), ('Alpha', 300)]
Top 2 by value           : [('Alpha', 300), ('Gamma', 200)]
Two-key sort             : [('Meera', 7), ('Ali', 5), ('Ravi', 5)]


### ⚡ Beyond the syllabus: `reduce`: running totals, maxima and cumulative growth

`reduce` folds a list down to one value by applying a two-argument function repeatedly: `reduce(add, [1,2,3])` computes `(1+2)+3`. You rarely need it for sums (`sum()` exists) but it's the right tool for compounding, running products, and merging dictionaries, cases with no built-in equivalent.

In [14]:
from functools import reduce

monthly_growth = [1.05, 1.03, 0.98, 1.07]     # +5%, +3%, -2%, +7%

# Compound growth — you cannot do this with sum()
compounded = reduce(lambda running, rate: running * rate, monthly_growth, 1.0)
print(f"Compounded growth over 4 months: {compounded:.4f}  ({(compounded-1)*100:+.2f}%)")

# reduce with an initial value (the 1.0 above) makes it safe on an empty list too
print("Empty list, no crash:", reduce(lambda a, b: a * b, [], 1.0))

# Merging dictionaries of stock counts from several warehouses
warehouses = [{'A': 10, 'B': 5}, {'A': 3, 'C': 8}, {'B': 2}]

def merge_counts(left, right):
    out = dict(left)
    for key, value in right.items():
        out[key] = out.get(key, 0) + value
    return out

print("Merged stock:", reduce(merge_counts, warehouses, {}))

# Running (cumulative) total — accumulate keeps every step, reduce keeps only the last
from itertools import accumulate
print("Running total:", list(accumulate([1200, 900, 1500, 2100])))

Compounded growth over 4 months: 1.1341  (+13.41%)
Empty list, no crash: 1.0
Merged stock: {'A': 13, 'B': 7, 'C': 8}
Running total: [1200, 2100, 3600, 5700]


## Part 3: `map` (Q11-Q15)

`map(function, list)` runs the function on every item. The catch that trips everyone: **`map` returns a lazy map object, not a list.** Wrap it in `list(...)` to see anything.

`map` and a comprehension do the same job. Use `map` when you already have a named function (`map(str.upper, names)` is beautifully clean); use a comprehension when the rule is inline.

**Q11. You have a list of product prices `[100, 200, 300]`. How can you use `map` to increase each price by 15%??**

In [15]:
# Solution for Q11
prices = [100, 200, 300]
updated_prices = list(map(lambda price: price * 1.15, prices))  # apply 15% hike
print(updated_prices)

[114.99999999999999, 229.99999999999997, 345.0]


**Q12. Given a list of employee working hours `[8, 7, 9]`, how can you use `map` to calculate overtime pay assuming 100 per hour??**

In [16]:
# Solution for Q12
hours = [8, 7, 9]
overtime_pay = list(map(lambda hrs: hrs * 100, hours))  # pay ₹100 per hour worked
print(overtime_pay)

[800, 700, 900]


**Q13. How can you use `map` with lambda to convert a list of product names to uppercase??**

In [17]:
# Solution for Q13
products = ['monitor', 'keyboard', 'mouse']
uppercase_products = list(map(lambda item: item.upper(), products))
print(uppercase_products)

['MONITOR', 'KEYBOARD', 'MOUSE']


**Q14. You have a list of revenue figures `[1000, 2000, 3000]`. How can you use `map` to calculate net revenue after 10% tax??**

In [18]:
# Solution for Q14
revenues = [1000, 2000, 3000]
net_profit = list(map(lambda amount: amount * 0.9, revenues))  # assume 10% tax
print(net_profit)

[900.0, 1800.0, 2700.0]


**Q15. Given a list of marks `[45, 78, 90]`, how can you use `map` to convert them into grades using a lambda??**

In [19]:
# Solution for Q15
marks = [45, 78, 90]
letter_grades = list(map(lambda score: 'A' if score >= 80 else 'B' if score >= 60 else 'C', marks))
print(letter_grades)

['C', 'B', 'A']


## Part 4: `filter` (Q16-Q20)

`filter(test, list)` keeps every item for which the test returns True. Same lazy-object catch: wrap in `list(...)`.

A useful mental translation: `filter(lambda x: x > 500, prices)` is `[x for x in prices if x > 500]`, and in pandas it becomes `prices[prices > 500]`. Three syntaxes, one idea.

**Q16. How can you use `filter` to find products above 500 from `[200, 450, 600, 750]`??**

In [20]:
# Solution for Q16
product_prices = [200, 450, 600, 750]
costly_products = list(filter(lambda price: price > 500, product_prices))
print(costly_products)

[600, 750]


**Q17. Given employee ages `[22, 34, 19, 45]`, how can you use `filter` to find only those above 30??**

In [21]:
# Solution for Q17
employee_ages = [22, 34, 19, 45]
adult_staff = list(filter(lambda age: age > 30, employee_ages))
print(adult_staff)

[34, 45]


**Q18. How can you use `filter` to extract transactions greater than 1000 from `[500, 1200, 2500, 800]`??**

In [22]:
# Solution for Q18
transactions = [500, 1200, 2500, 800]
large_transactions = list(filter(lambda amount: amount > 1000, transactions))
print(large_transactions)

[1200, 2500]


**Q19. Given a list of customers `['Sam', 'Anna', 'Sophia', 'Raj']`, how can you use `filter` to select names starting with 'S'??**

In [23]:
# Solution for Q19
customers = ['Sam', 'Anna', 'Sophia', 'Raj']
selected_customers = list(filter(lambda name: name.startswith('S'), customers))
print(selected_customers)

['Sam', 'Sophia']


**Q20. How can you use `filter` with lambda to keep only even invoice numbers `[101, 102, 103, 104]`??**

In [24]:
# Solution for Q20
invoices = [101, 102, 103, 104]
even_invoices = list(filter(lambda invoice: invoice % 2 == 0, invoices))
print(even_invoices)

[102, 104]


## Part 5: Combining them (Q21-Q35)

The rest of the questions mix all three plus sorting. The one to study hardest is **Q28, sorting with `key=lambda`**. It's the pattern behind "top 5 products by revenue", which turns up in almost every exam.

**Q21. Using list comprehension, how can you compute final prices by adding 18% GST to `[100, 200, 300]`??**

In [25]:
# Solution for Q21
prices = [100, 200, 300]
final_prices = [price * 1.18 for price in prices]  # add 18% GST inline
print(final_prices)

[118.0, 236.0, 354.0]


**Q22. How can you use `map` and `lambda` together to compute square of `[5, 10, 15]`??**

In [26]:
# Solution for Q22
values = [5, 10, 15]
squares = list(map(lambda value: value ** 2, values))
print(squares)

[25, 100, 225]


**Q23. Using `filter` and `lambda`, how can you select salaries greater than 50k from `[40000, 55000, 60000, 30000]`?**

In [27]:
# Solution for Q23
salaries = [40000, 55000, 60000, 30000]
high_salaries = list(filter(lambda salary: salary > 50000, salaries))
print(high_salaries)

[55000, 60000]


**Q24. How can you combine `map` and `filter` to first increase all prices by 10% and then select only those above 200 from `[100, 150, 250]`?**

In [28]:
# Solution for Q24
prices = [100, 150, 250]
adjusted = list(map(lambda price: round(price * 1.10, 2), prices))
premium_only = list(filter(lambda price: price > 200, adjusted))
print("Adjusted prices:", adjusted)
print("Premium selections (> 200):", premium_only)

Adjusted prices: [110.0, 165.0, 275.0]
Premium selections (> 200): [275.0]


**Q25. Using list comprehension, how can you replace all negative values with 0 in `[10, -5, 20, -8]`??**

In [29]:
# Solution for Q25
values = [10, -5, 20, -8]
cleaned_values = [value if value > 0 else 0 for value in values]
print(cleaned_values)

[10, 0, 20, 0]


**Q26. How can you use `map` to format a list of invoice numbers `[1, 2, 3]` as `INV-1, INV-2, INV-3`??**

In [30]:
# Solution for Q26
invoices = [1, 2, 3]
formatted_invoices = list(map(lambda number: f"INV-{number}", invoices))
print(formatted_invoices)

['INV-1', 'INV-2', 'INV-3']


**Q27. Using `filter`, how can you extract only profitable transactions from `[100, -50, 200, -30]`??**

In [31]:
# Solution for Q27
transactions = [100, -50, 200, -30]
profitable_transactions = list(filter(lambda amount: amount > 0, transactions))
print(profitable_transactions)

[100, 200]


**Q28. How can you use lambda in sorting a list of tuples `[('A', 300), ('B', 100), ('C', 200)]` by the second element??**

In [32]:
# Solution for Q28
sales = [('A', 300), ('B', 100), ('C', 200)]
ranked_sales = sorted(sales, key=lambda record: record[1], reverse=True)  # sort by sales descending
print(ranked_sales)

[('A', 300), ('C', 200), ('B', 100)]


**Q29. Using list comprehension, how can you flatten `[[1,2],[3,4],[5,6]]` into `[1,2,3,4,5,6]`??**

In [33]:
# Solution for Q29
nested = [[1, 2], [3, 4], [5, 6]]
flat = [item for sublist in nested for item in sublist]
print(flat)

[1, 2, 3, 4, 5, 6]


In [34]:
# --- Another valid way to write the same answer -------------------
# WHY show two? Examiners give credit for either. Pick whichever you
# can reproduce from memory under time pressure.
to_flatten = [[1,2],[3,4],[5,6]]
after_flattening = [j for x in to_flatten for j in x]
after_flattening

[1, 2, 3, 4, 5, 6]

**Q30. How can you use `map` and `lambda` to convert `[10, 20, 30]` into their string equivalents??**

In [35]:
# Solution for Q30
numbers = [10, 20, 30]
as_strings = list(map(lambda number: str(number), numbers))
print(as_strings)

['10', '20', '30']


**Q31. Using `filter` and `lambda`, how can you select all words longer than 5 characters from `['apple', 'banana', 'kiwi', 'watermelon']`??**

In [36]:
# Solution for Q31
words = ['apple', 'banana', 'kiwi', 'watermelon']
long_words = list(filter(lambda word: len(word) > 5, words))
print(long_words)

['banana', 'watermelon']

**Q32. How can you use list comprehension to extract the first character from each word in `['Sales', 'Marketing', 'Finance']`??**

In [37]:
# Solution for Q32
words = ['Sales', 'Marketing', 'Finance']
initials = [word[0] for word in words]
print(initials)

['S', 'M', 'F']


**Q33. Using `map` and `lambda`, how can you calculate square roots of `[4, 9, 16]`??**

In [38]:
# Solution for Q33
import math
values = [4, 9, 16]
square_roots = list(map(lambda number: math.sqrt(number), values))
print(square_roots)

[2.0, 3.0, 4.0]


**Q34. How can you use `filter` and `lambda` to select only positive numbers from `[-10, 0, 5, 15]`??**

In [39]:
# Solution for Q34
values = [-10, 0, 5, 15]
positives = list(filter(lambda value: value > 0, values))
print(positives)

[5, 15]


**Q35. Using list comprehension, how can you generate all possible pairs from `[1,2]` and `[3,4]`??**

In [40]:
# Solution for Q35
first = [1, 2]
second = [3, 4]
pairs = [(a, b) for a in first for b in second]
print(pairs)

[(1, 3), (1, 4), (2, 3), (2, 4)]


### ⚡ Beyond the syllabus: the same five operations in pure Python vs pandas

Everything in this notebook has a pandas twin, and from Section 02 onwards the pandas version is what you'll be marked on. Keep this translation table where you can see it; it converts any lambda/map/filter answer into the DataFrame answer.

In [41]:
import pandas as pd

prices = [100, 200, 300, 450, 600]
s = pd.Series(prices, name='price')

print("TASK                    PURE PYTHON                              PANDAS")
print("-" * 88)
print(f"add 18% GST            {str(list(map(lambda p: p*1.18, prices))):<40} {list(s * 1.18)}")
print(f"keep above 250         {str(list(filter(lambda p: p > 250, prices))):<40} {list(s[s > 250])}")
print(f"label High/Low         {str(['H' if p>250 else 'L' for p in prices]):<40} "
      f"{list(s.where(s <= 250, 'H').mask(s > 250, 'H').pipe(lambda x: ['H' if v>250 else 'L' for v in s]))}")
print(f"total                  {str(sum(prices)):<40} {s.sum()}")
print(f"sort descending        {str(sorted(prices, reverse=True)):<40} {list(s.sort_values(ascending=False))}")

print("\nThe cleanest pandas idioms for each:")
print("  transform :  s * 1.18            or  s.apply(lambda p: p * 1.18)")
print("  filter    :  s[s > 250]          <- boolean mask, the pandas 'filter'")
print("  label     :  pd.cut / np.select  <- see 04_Data_Transformation")
print("  aggregate :  s.sum(), s.mean(), s.max()")
print("  sort      :  s.sort_values(ascending=False).head(5)   <- 'top 5' questions")

TASK                    PURE PYTHON                              PANDAS
----------------------------------------------------------------------------------------
add 18% GST            [118.0, 236.0, 354.0, 531.0, 708.0]      [118.0, 236.0, 354.0, 531.0, 708.0]
keep above 250         [300, 450, 600]                          [300, 450, 600]
label High/Low         ['L', 'L', 'H', 'H', 'H']                ['L', 'L', 'H', 'H', 'H']
total                  1650                                     1650
sort descending        [600, 450, 300, 200, 100]                [600, 450, 300, 200, 100]

The cleanest pandas idioms for each:
  transform :  s * 1.18            or  s.apply(lambda p: p * 1.18)
  filter    :  s[s > 250]          <- boolean mask, the pandas 'filter'
  label     :  pd.cut / np.select  <- see 04_Data_Transformation
  aggregate :  s.sum(), s.mean(), s.max()
  sort      :  s.sort_values(ascending=False).head(5)   <- 'top 5' questions


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| One-line function | `f = lambda x: x * 1.18` |
| Two arguments | `lambda a, b: a - b` |
| Conditional inside a lambda | `lambda x: 'High' if x>500 else 'Low'` |
| Apply to every item | `list(map(f, lst))` |
| Apply a built-in method | `list(map(str.upper, names))` |
| Keep matching items | `list(filter(lambda x: x>500, lst))` |
| Combine both | `list(filter(t, map(f, lst)))` |
| Fold to one value | `reduce(lambda a,b: a+b, lst)` |
| Sort by 2nd element | `sorted(pairs, key=lambda t: t[1])` |
| Top 5 by value | `sorted(d.items(), key=lambda kv: -kv[1])[:5]` |
| Sort a dict by value | `dict(sorted(d.items(), key=lambda kv: kv[1]))` |
| Numbers → text | `list(map(str, nums))` |
| Square roots | `list(map(lambda x: x**0.5, nums))` |

### Adapting this in the exam

- Question says 'apply to each' → `map` or a comprehension.
- Question says 'select/keep only/where' → `filter` or a comprehension with `if`.
- Question says 'total/combine/accumulate' → `sum`, or `reduce` if it isn't a plain sum.
- Question says 'top N' or 'rank' → `sorted(..., key=lambda …, reverse=True)[:N]`.

### Traps that cost marks

- `map` and `filter` return **lazy objects**. `print(map(...))` shows `<map object …>`. Always wrap in `list()`.
- A lazy object is **consumed once**. Loop over it twice and the second pass is empty; convert to a list first if you need it again.
- A lambda holds one expression only. Anything needing a loop, a variable assignment or multiple lines must be a `def`.
- `sorted(...)` returns a new list; `lst.sort()` returns `None` and sorts in place. Mixing them up prints `None`.
- `key=lambda t: t[1]` sorts **by** the second element but still returns whole tuples, which is usually what you want.
- `reduce` must be imported: `from functools import reduce`. It is not built in.